# 18. Lecke: AI ügynökök biztosítása kriptográfiai bizonylatokkal

## Gyakorlati jegyzetfüzet

Ez a jegyzetfüzet négy gyakorlatot mutat be:

1. **Írd alá az első bizonylatodat** egy ügynök eszköz hívásához, és ellenőrizd azt.
2. **Manipulálj a bizonylaton**, és figyeld meg, hogyan sikertelen az ellenőrzés.
3. **Építs három bizonylatból álló láncot**, és erősítsd meg a lánc integritását.
4. **Csomagolj be egy Microsoft Agent Framework eszköz hívást**, hogy minden művelet bizonylatot bocsásson ki.

Minden kriptográfiai primitív jól karbantartott könyvtárakból importált (Ed25519-hez `pynacl`, RFC 8785 kompatibilis kanonikus JSON-hoz `jcs`, SHA-256-hoz a Python szabványos könyvtárából `hashlib`). Maga a bizonylat logika egyszerű Python, amit elolvashatsz és módosíthatsz.

Futtasd a cellákat sorrendben. Minden szakasz rövid és önálló.


## Beállítás

Telepítse a két függőséget. Mindkettő engedékeny licenccel rendelkezik (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Segéd segédprogramok

Ezek a két segéd a base64url kódolást (kitöltés nélkül) és a tetszőleges objektumok SHA-256 hashelését végzik. Ezek segítenek abban, hogy a jegyzet többi része kizárólag a bizonylat logikájára összpontosítson.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## 1. szakasz: Írd alá az első blokklánc-nyugtádat

Képzeljük el, hogy a **Contoso Travel** ügynökünktől egy ügyfél Sydneyből Los Angelesbe keresett repülőjáratokat. Szeretnénk ezt az eszközhívást aláírt nyugtaként rögzíteni, hogy egy későbbi ellenőr anélkül ellenőrizhesse, hogy nekünk kelljen bíznia benne.

### 1.1 lépés: Aláíró kulcs létrehozása

Éles környezetben az ügynök aláíró kulcsa hardveres biztonsági modulban (HSM), Azure Key Vaultban vagy hasonló védett tárolóban lenne elhelyezve. Ebben a leckében most egy friss kulcsot generálunk memóriában.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### 1.2. lépés: A bizonylat terhelésének összeállítása

A terhelés tartalmaz mindent, amit a bizonylatnak igazolnia kell: ki cselekedett, milyen eszköz, milyen argumentumokkal, mi tért vissza, milyen szabályzat alatt, és mikor. Az argumentumokat és az eredményt helyben történő megjelenítés helyett hash-eljük, hogy a bizonylat ne szivárogtasson érzékeny tartalmat.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### 1.3 lépés: A nyugta aláírása és összeállítása

Három lépés:

1. A terhelés kanonizálása JCS segítségével, hogy két ugyanazt a logikai nyugtát előállító megvalósítás bájtra pontosan azonos bájtokat adjon.
2. A kanonikus bájtok SHA-256-tal való összegzése.
3. A hash aláírása Ed25519 privát kulccsal.

Az aláírást ezután az eredeti terheléshez csatolják, hogy létrehozzák a végső nyugtát.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### 1.4 lépés: A nyugta ellenőrzése

Az ellenőrzés megfordítja a folyamatot. Eltávolítjuk az aláírást, újraszámoljuk a kanonikus hasht, és ellenőrizzük az aláírást a nyugtában található nyilvános kulccsal.

Egy ellenőr, aki ezt az ellenőrzést végzi, semmi mást nem igényel tőlünk, csak magát a nyugtát. Nincs szükség szolgáltatás hívására, kulcskönyvtár lekérdezésére vagy bizalomra.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Látnod kell, hogy `Receipt is valid: True`. Az ügynök elkészítette az első kriptográfiailag aláírt audit naplóját.


## 2. fejezet: A blokk eltulajdonítása

A blokkok lényege, hogy észlelhető legyen az eltulajdonításuk. Bizonyítsuk be ezt.

Pontosan az egyik karaktert fogjuk módosítani a blokkon, és figyeljük meg az ellenőrzés meghiúsulását.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Mi történt éppen?

Amikor megváltoztattuk a `policy_id` értékét, megváltoztak a kanonikus bájtok. Ezeknek a bájtoknak a SHA-256 hash értéke is megváltozott. Az aláírás (ami az eredeti hash felett készült) már nem egyezik az új hash értékkel. Az ellenőrzés helyesen ad vissza `False` értéket.

Nincs mód bármely mező megváltoztatására a bizonylaton úgy, hogy az még érvényes legyen, kivéve, ha a támadónak megvan a privát kulcs. Amíg a privát kulcs kulcs tárolóban van, és a nyilvános kulcs nyilvánosan elérhető, a manipuláció elrejtése lehetetlen.

Próbáld ki magad: módosítsd a fenti cellában a `tool_name`-t, `agent_id`-t vagy `timestamp`-et, majd futtasd újra. Minden változtatás érvénytelen bizonylatot eredményez.


## 3. fejezet: Blokkok láncolása egymáshoz

Egyetlen blokk egyetlen műveletet véd. A legtöbb ügynök sok műveletet végez. Ahhoz, hogy az egész sorozat manipulálás ellen védett legyen, minden blokkot az előzőhöz láncolunk úgy, hogy a korábbi blokk hash-értékét beillesztjük az új blokk adattermékébe.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Ha valaki eltávolít vagy átstrukturál egy blokkot, a lánc pontosan ott megszakad. Bármely későbbi blokk érvényesítése meghiúsul, mert annak `previous_receipt_hash` értéke nem egyezik meg a tényleges előd hash-értékével.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Most törje meg a láncot a középső bizonylat megváltoztatásával, majd ellenőrizze újra. A módosított bizonylat meghiúsítja az aláírás-ellenőrzését, ÉS a következő bizonylat meghiúsítja a lánc-láncellenőrzést (mivel a `previous_receipt_hash` már nem egyezik a módosított középső bizonylat hash-ével).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

A 0. bizonylat továbbra is ellenőrizhető (nem módosították, és nincs előzménye, amelytől függne). Az 1. bizonylat aláírás-ellenőrzése meghiúsul, mert megváltoztattuk a `tool_args_hash` értékét. A 2. bizonylat lánchivatkozás ellenőrzése meghiúsul, mert az ő `previous_receipt_hash` értéke az eredeti (most módosított) 1. bizonylatra készült.

Még ha egy támadó újra alá is írnád a módosított 1. bizonylatot (amit a privát kulcs nélkül nem tehet meg), a 2. bizonylat lánchivatkozás-eltérése továbbra is felfedné a manipulációt. A változás elrejtéséhez a támadónak minden bizonylatot újra kellene aláírnia a módosítás helyétől kezdve, amihez a privát kulcs birtoklása szükséges.


## 4. szakasz: Egy ügynök eszközhívás burkolása átvételi aláírással  

Egy valódi telepítésnél nem szeretnéd, hogy minden ügynökszerzőnek meg kelljen jegyeznie a `make_receipt` hívását. Az átvételi aláírásnak automatikusnak kell lennie minden eszköz meghívásakor.  

Itt a legegyszerűbb minta: egy burkoló osztály, amely bármely hívható eszközfüggvényt átvesz, és átvételt kibocsátó változatát adja vissza. Ez bármely ügynök keretrendszerhez alkalmazkodik, beleértve a Microsoft Agent Framework-öt (`agent_framework.foundry`).  

Ha nincs felállítva Microsoft Foundry projekted, az alábbi helyi próbaverzió is bemutatja a mintát.  


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Integráció a Microsoft Agent Framework-kel

A fenti `ReceiptedTool` csomagoló keretfüggetlen. Ha Microsoft Agent Framework-kel épített ügynökön belül szeretnéd használni, regisztráld a becsomagolt függvényt eszközként. Egy vázlat (a mock-ot valódi Microsoft Foundry eszközregisztrációra kell cserélni):

```python
# Pseudokód az integrációs forma bemutatására.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="Ön egy Contoso Travel ügynök ...",
#     tools=[receipted_lookup],   # a becsomagolt eszköz, nem a nyers függvény
# )
# response = agent.run("Keress repülőjáratokat Sydneyből Los Angelesbe júniusban.")
#
# # A futtatás után minden eszközhasználat, amit az ügynök végrehajtott, aláírt nyugtával rendelkezik:
# audit_chain = receipted_lookup.receipts
```

Az agent framework-nek nem kell tudnia semmit a nyugtákról. A nyugták aláírása az eszköz köré van csomagolva, nem pedig a framework-be beépítve. Így adhatsz eredetséget a meglévő agent kódhoz anélkül, hogy át kellene írni az agentet.


## Összefoglalás és extra kihívás

Eddig megtetted:

- Egy Ed25519 kulcspár generálása.
- Egy ügynök eszközhívásának bizonylatának létrehozása és aláírása.
- A bizonylat offline ellenőrzése kizárólag a nyilvános kulcs használatával.
- Egy bizonylat manipulálása és az ellenőrzési sikertelenség megfigyelése.
- Három bizonylatból álló hash-láncolt sorozat létrehozása.
- A lánc közepének manipulálása és az aláírási, valamint a lánclánc hiba megfigyelése.
- Egy eszköz függvényének automatikus bizonylat aláírással való becsomagolása.

**Extra kihívás.** Bővítsd a bizonylat sémát egy `request_id` mezővel (egy UUID az elosztott követéshez). Frissítsd a `make_receipt`-et, hogy tartalmazza ezt, és erősítsd meg, hogy a bizonylatok továbbra is végeznek végponti ellenőrzést. Ezután módosítsd a mezőt az aláírás után, és erősítsd meg, hogy az ellenőrzés sikertelen lesz. Ez arra kényszerít, hogy belsőleg megértsd, hogyan járul hozzá a kanonikus kódolás minden bájtja az aláíráshoz.

**Fontos határvonal.** A bizonylatok három dolgot bizonyítanak, és csak három dolgot: hozzárendelés (ez a kulcs írta alá ezt a tartalmat), integritás (a tartalom nem változott az aláírás óta), és sorrend (ez a bizonylat később érkezett, mint az a bizonylat). Nem bizonyítják, hogy az ügynök művelete helyes volt, hogy a `policy_id`-ban megnevezett szabályzat valóban ki lett-e értékelve, vagy hogy az ügynök minden szabályt betartott. A bizonylatok alapot szolgáltatnak. A kormányzás az a rendszer, amit erre építesz.

Olvasd el újra a lecke README-jét ezzel a határvonallal a szemed előtt. A leggyakoribb hiba, amit a csapatok a bizonylatokkal kapcsolatban elkövetnek, hogy azt feltételezik, "van bizonylatunk" azt jelenti, hogy "rákormányzunk". Ez nem igaz. A bizonylatok lehetővé teszik az ügynök viselkedésének auditálását. Nem teszik helyessé azt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
